# 07 — Synthetic galaxies and their star-formation histories

*Lecture 1 (stellar populations) → Lecture 2 (star-formation histories, the [Mg/Fe] clock)*

A galaxy is a **sum of SSPs**: every generation of stars, with its own age and metallicity, adds its spectrum weighted by the **mass it formed**:

$$F_\mathrm{gal}(\lambda) = \Big[\sum_{i} M_i\, F_\mathrm{SSP}(\lambda;\, t_i, Z_i)\Big] \ast \mathrm{LOSVD} + \text{noise}.$$

Recovering the weights $M_i \ge 0$ is a **linear** problem with a positivity constraint (non-negative least squares). It is ill-posed: many SFHs fit the data equally well, so we add **regularisation** (prefer smooth solutions).

We use **pPXF** (Cappellari & Emsellem 2004; Cappellari 2017, 2023) — the standard tool for this — with FaSTAR templates.

**Extra packages:** `pip install ppxf` (pPXF; numpy/scipy/matplotlib only) and `pip install fastar-astro`.
Alternatives you may meet: Prospector, FAST, BAGPIPES, STARLIGHT, FIREFLY, pyPipe3D.

⏱ The template grid takes ~1 minute, each fit a few seconds.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Azeret/stellar-populations-lectures/blob/main/notebooks/07_SFH_of_synthetic_galaxies.ipynb)  ← run this notebook in your browser, nothing to install

In [ ]:
# Running in Google Colab? This cell fetches the course code and installs fastar-astro ppxf.
# On your own computer it does nothing.
import sys, os, subprocess
if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/stellar-populations-lectures'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Azeret/stellar-populations-lectures.git',
                        '/content/stellar-populations-lectures'], check=True)
    os.chdir('/content/stellar-populations-lectures/notebooks')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastar-astro', 'ppxf'], check=True)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # makes the small 'stellarpops' package importable
import numpy as np
import matplotlib.pyplot as plt
from stellarpops.style import use_slide_style
use_slide_style(dark=False, scale=0.7)       # dark=True gives the lecture-slide look

from fastar import IntegratedSSPSynthesizer, imf
from ppxf.ppxf import ppxf
from scipy.optimize import nnls
from stellarpops.spectra import log_rebin, broaden_velocity, add_noise

ssp = IntegratedSSPSynthesizer(imf_function=imf.kroupa)
wave = np.array(ssp.wave)

## 1. Templates: a grid of SSPs

FaSTAR spectra are *per solar mass formed*, so the pPXF weights will be **masses formed**. We log-rebin all templates (notebook 05).

In [ ]:
ages = np.geomspace(0.06, 13.0, 14)               # Gyr
mets = np.array([-1.5, -1.0, -0.5, 0.0, 0.25])    # [M/H]
grid = np.array([[np.array(ssp.synthesize(age=float(a), met=float(m))) for m in mets] for a in ages])
SCALE = np.median(grid)                           # keep numbers near 1

ln_lam, _, velscale = log_rebin(wave, grid[0, 0])
templates = np.array([[np.interp(ln_lam, np.log(wave), grid[i, j]) for j in range(mets.size)]
                      for i in range(ages.size)]) / SCALE
templates = np.moveaxis(templates, -1, 0)         # pPXF wants (n_pixels, n_ages, n_mets)
print('templates:', templates.shape, f' velscale = {velscale:.1f} km/s')

## 2. Four synthetic galaxies

We build them from a **finer** set of ages than the template grid (fitting with exactly the same templates that made the mock would be cheating — the "inverse crime").

In [ ]:
def make_galaxy(sfh, zfun, sigma, snr, seed=0, t_max=13.0, n=60):
    # sfh(t): star-formation rate at lookback time t [Gyr]; zfun(t): [M/H] of stars born then
    t = np.geomspace(0.05, t_max, n)
    dt = np.gradient(t)
    mass = sfh(t) * dt
    spec = sum(m * np.array(ssp.synthesize(age=float(a), met=float(np.clip(zfun(a), -2.4, 0.29))))
               for a, m in zip(t, mass) if m > 0) / SCALE
    lnl, s, _ = log_rebin(wave, spec, velscale)
    s = broaden_velocity(s, velscale, sigma)
    s, noise = add_noise(s, snr, rng=seed)
    return dict(t=t, mass=mass / mass.sum(), z=zfun(t), lam=np.exp(lnl), flux=s, noise=noise, sigma=sigma)

galaxies = {
  'old elliptical':  make_galaxy(lambda t: np.exp(-0.5 * ((t - 11.5) / 0.7) ** 2), lambda t: 0.1 + 0 * t, sigma=250, snr=60),
  'Milky-Way disc':  make_galaxy(lambda t: (t < 11) * 1.0, lambda t: 0.1 - 0.08 * t, sigma=100, snr=60),
  'post-starburst':  make_galaxy(lambda t: np.exp(-0.5 * ((t - 11) / 1) ** 2) + 3 * np.exp(-0.5 * ((t - 0.6) / 0.1) ** 2),
                                 lambda t: 0.0 + 0 * t, sigma=150, snr=60),
  'bursty dwarf':    make_galaxy(lambda t: np.exp(-0.5 * ((t - 9) / 0.8) ** 2) + 0.6 * np.exp(-0.5 * ((t - 1.5) / 0.3) ** 2),
                                 lambda t: np.where(t > 4, -1.0, -0.6), sigma=40, snr=60),
}
fig, ax = plt.subplots(figsize=(11, 4.5))
for k, (name, g) in enumerate(galaxies.items()):
    sel = (g['lam'] > 3800) & (g['lam'] < 7200)
    ax.plot(g['lam'][sel], g['flux'][sel] / np.median(g['flux'][sel]) + 0.6 * k, lw=0.6, label=name)
ax.set_xlabel(r'wavelength [$\AA$]'); ax.legend(fontsize=9, loc='upper right')

## 3. Fit one galaxy with pPXF

* `start=[V, sigma]` — first guess of the kinematics (km/s); `moments=2` fits V and σ.
* `degree=-1, mdegree=8` — no additive polynomial; a multiplicative one absorbs flux-calibration errors and dust.
* `regul` — regularisation strength (0 = plain non-negative least squares).

In [ ]:
def fit(g, regul=0, mdegree=8, plot=False):
    sel = (g['lam'] > 3800) & (g['lam'] < 7200)
    pp = ppxf(templates, g['flux'][sel], g['noise'][sel], velscale, start=[0, 100], moments=2,
              degree=-1, mdegree=mdegree, lam=g['lam'][sel], lam_temp=np.exp(ln_lam),
              regul=regul, reg_dim=templates.shape[1:], quiet=True, plot=plot)
    w = pp.weights.reshape(ages.size, mets.size)
    return pp, w / w.sum()

g = galaxies['post-starburst']
pp, w = fit(g, regul=0, plot=True)
print(f'sigma: fitted {pp.sol[1]:.0f} km/s, true {g["sigma"]} km/s;  chi2/pixel = {pp.chi2:.2f}')

## 4. Recovered vs true star-formation history

We compare the **cumulative** mass formed (from the oldest stars to today), which is much more robust than the mass in single age bins.

In [ ]:
def compare(name, regul_values=(0, 30, 300)):
    g = galaxies[name]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
    order = np.argsort(g['t'])[::-1]
    a2.plot(g['t'][order], np.cumsum(g['mass'][order]), 'k', lw=4, label='truth')
    for regul in regul_values:
        pp, w = fit(g, regul=regul)
        m_age = w.sum(axis=1)
        a1.plot(ages, m_age, 'o-', label=f'regul = {regul}')
        a2.plot(ages[::-1], np.cumsum(m_age[::-1]), 'o-', label=f'regul = {regul}')
        mw_age = 10 ** (m_age @ np.log10(ages)); mw_z = w.sum(axis=0) @ mets
        print(f'regul={regul:4d}: mass-weighted age {mw_age:5.2f} Gyr, [M/H] {mw_z:+.2f}  (chi2/pix {pp.chi2:.2f})')
    print(f'truth     : mass-weighted age {10 ** (g["mass"] @ np.log10(g["t"])):5.2f} Gyr, [M/H] {g["mass"] @ g["z"]:+.2f}')
    for ax in (a1, a2):
        ax.set_xscale('log'); ax.set_xlabel('lookback time (age) [Gyr]')
    a1.set_ylabel('fraction of mass formed'); a2.set_ylabel('cumulative mass fraction (oldest first)')
    a1.set_title(name); a2.legend(fontsize=9)

compare('post-starburst')

In [ ]:
compare('old elliptical')

In [ ]:
compare('Milky-Way disc')

In [ ]:
compare('bursty dwarf')

**Questions.**
* Without regularisation (`regul=0`) the solution is spiky: why does a least-squares fit love a few templates?
* Which features are recovered robustly: the young burst, the old bulk, the exact ages?
* Compare the mass-weighted [M/H] with the truth. Where does the age–metallicity degeneracy show up?

## 5. What pPXF does, by hand

With the kinematics fixed, the problem is **non-negative least squares**: find $w_i \ge 0$ minimising $\|(\mathbf{T} w - F)/\sigma\|^2$. `scipy.optimize.nnls` solves it in one line.

In [ ]:
g = galaxies['post-starburst']
sel = (g['lam'] > 3800) & (g['lam'] < 7200)
# templates on the galaxy's wavelength pixels, broadened with the true sigma
T = np.array([broaden_velocity(templates[:, i, j], velscale, g['sigma'])
              for i in range(ages.size) for j in range(mets.size)]).T
T = np.array([np.interp(np.log(g['lam'][sel]), ln_lam, T[:, k]) for k in range(T.shape[1])]).T
A = T / g['noise'][sel, None]; b = g['flux'][sel] / g['noise'][sel]
w_nnls, _ = nnls(A, b)
w_nnls = w_nnls.reshape(ages.size, mets.size); w_nnls /= w_nnls.sum()
print('templates used (weight > 1%):', int((w_nnls > 0.01).sum()), 'of', w_nnls.size)
plt.plot(ages, w_nnls.sum(1), 'o-', label='scipy nnls'); plt.plot(ages, fit(g, regul=0)[1].sum(1), 's--', label='pPXF, regul=0')
plt.xscale('log'); plt.xlabel('age [Gyr]'); plt.ylabel('fraction of mass formed'); plt.legend()

## Exercises

1. Lower the S/N of the post-starburst to 15 (`make_galaxy(..., snr=15)`). Is the young burst still found? Its mass? Its age?
2. Make the burst **older** (t = 2 Gyr instead of 0.6 Gyr). Why is it harder to recover?
3. Add 30% more "dust": multiply the galaxy by a smooth slope, `g['flux'] *= (g['lam'] / 5500) ** 0.5`. Fit with `mdegree=0` and `mdegree=8`. What happens?
4. Build templates with a top-heavy or bottom-heavy IMF (`imf.single_power_law`, `imf_params={'alpha': ...}`) and fit the Milky-Way disc. How much does the inferred SFH depend on the IMF you assume?
5. Lecture 2 link: the SFH of the old elliptical tells you its star-formation **duration**. Compare it with the [Mg/Fe] ≈ +0.3 argument (Slido S2.9).